## **MLlib: 영화 추천 알고리즘 구현**
+ 데이터셋: MovieLens(2500만개 영화 평점 데이터), 본 예제에서는 7만개의 데이터만 사용 ```ratings_short.csv```
+ 추천 알고리즘: Alternating Least Squares (ALS)

---

### **1. 영화 평점 데이터 불러오기 & 데이터프레임 생성하기**

In [ ]:
# [+] SparkSession 설정
...

#### movielens 데이터 불러오기
+ 파일명: ```ratings_short.csv```
+ 스키마 설정: ```inferSchema=True```
+ 헤더 사용: ```header=True```

In [ ]:
# movielens 데이터 불러오기
path = '../data/'
file = 'ratings_short.csv'

ratings_df = spark.read.csv(
    path + file, 
    inferSchema=True, 
    header=True
)

In [ ]:
# 데이터프레임 출력
ratings_df.show()

In [ ]:
# 데이터프레임 스키마 출력
ratings_df.printSchema()

In [ ]:
# [+] 타임스탬프를 제외한 컬럼 선택
ratings_df = ...

In [ ]:
# [+] describe(): 기술 통계량 출력
ratings_df...

### **2. 훈련 데이터 준비 및 영화 추천 모델 학습**

In [ ]:
# [+] 훈련 데이터셋과 테스트 데이터셋 분리 (8:2)
train_df, test_df = ...

In [ ]:
# 추천 알고리즘(Alternating Least Squares) 불러오기
from  pyspark.ml.recommendation import ALS

#### **ALS 알고리즘 초매개변수**
+ ```maxIter```: 최대 학습 반복 횟수
+ ```regParam```: 정규화 매개변수(범위: 0~1)
+ ```coldStartStrategy```: 데이터가 부족한 신규 유저 및 아이템에 대한 예측 문제(Cold Start)를 처리하는 방식이며 ```drop```값은 해당 데이터를 모델 학습 과정에서 배제

In [ ]:
# 추천 알고리즘 초매개변수 설정
als = ALS(
    maxIter=5,
    regParam=0.1,
    userCol='userId',
    itemCol='movieId',
    ratingCol='rating',
    coldStartStrategy='drop'
)

In [ ]:
# [+] 모델 학습
model = ...

In [ ]:
# # 메모리 부족으로 인한 오류 발생시, 아래의 코드를 실행
# from pyspark.sql import SparkSession

# MAX_MEMORY = '5g'
# spark = SparkSession.builder.appName('movie-recommendation')\
#     .config('spark.executor.memory', MAX_MEMORY)\
#     .config('spark.driver.memory', MAX_MEMORY)\
#     .getOrCreate()

In [ ]:
# [+] 모델 예측
predictions = ...

In [ ]:
# 예측값 출력
predictions.show()

In [ ]:
# [+] 평점과 예측평점에 대한 통계 출력
predictions...

In [ ]:
# 모델 성능 평가: RMSE(Root Mean Squared Error)
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='rating',
    predictionCol='prediction'
)

In [ ]:
# [+] RMSE 측정
rmse = ...

In [ ]:
rmse

### **3. 학습된 모델을 이용한 영화 추천**
+ ```recommendForAllUsers()```: 유저별 아이템 추천
+ ```recommendForAllItems()```: 아이템별 유저 추천
+ ```recommendForUserSubset()```: 특정 유저 그룹에 대한 아이템 추천

In [ ]:
# [+] 유저별 아이템을 3개씩 추천
model...

In [ ]:
# [+] 아이템별 유저를 3명씩 추천
model...

**특정 유저 그룹에 대한 아이템 추천**
1. 해당 유저들의 ID 정보를 데이터프레임으로 생성
2. ```recommendForUserSubset()```를 이용한 아이템 추천

In [ ]:
# 특정 유저 ID 집합: 2차원 리스트
user_lst = [[1]]

In [ ]:
# 스키마 정의
from pyspark.sql.types import StructType, StructField, IntegerType
schema = StructType([
    StructField('userID', IntegerType(), True)
])

In [ ]:
# 데이터프레임 생성
users_df = spark.createDataFrame(user_lst, schema)

In [ ]:
users_df.show()

In [ ]:
# [+] recommendForUserSubset(): 특정 유저 그룹에 대한 아이템 추천
user_recs = ...

In [ ]:
user_recs.show()

In [ ]:
# 추천 결과를 Python 객체로 받아오기
movies_lst = user_recs.collect()[0].recommendations

In [ ]:
movies_lst

In [ ]:
# movies_lst에 대한 데이터프레임 생성
recs_df = spark.createDataFrame(movies_lst)
recs_df.show()

In [ ]:
# [+] 영화 데이터에 대한 데이터프레임 생성

file = 'movies.csv'

movies_df = ...
movies_df.show()

In [ ]:
# [+] recs_df, movies_df 에 대한 Temporary View 생성
recs_df...
movies_df...

In [ ]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
spark.sql("""
    SELECT *
    FROM movies JOIN recommendations
    ON movies.movieID = recommendations.movieID
    ORDER BY rating DESC
""").show()

### **4. 유저 별 영화 추천 서비스를 간단하게 구현하기**
1. SQL문 작성
2. 영화 추천 함수 작성
3. 영화 추천 테스트

In [ ]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
query = """
SELECT * 
FROM movies JOIN recommendations 
ON movies.movieID = recommendations.movieID
ORDER BY rating DESC
"""

In [ ]:
# 입력된 유저에 대한 영화 추천 함수
def get_recommendations(user_id, num_recs):
    users_df = spark.createDataFrame([user_id], IntegerType()).toDF('userID')
    users_recs_df = model.recommendForUserSubset(users_df, num_recs)
    recs_lst = users_recs_df.collect()[0].recommendations
    recs_df = spark.createDataFrame(recs_lst)
    recs_df.createOrReplaceTempView('recommendations')
    recommended_movies = spark.sql(query)
    
    return recommended_movies

In [ ]:
# 1번 유저에 대한 영화 5개 추천
recs = get_recommendations(1, 5)

In [ ]:
# 추천 결과 출력
recs.show()

In [ ]:
# toPandas(): Pandas 데이터프레임으로 출력
recs.toPandas()